# IC Approximation Benchmarks

This notebook runs the final synthetic and real-world benchmarks using the cleaned repository layout.

Expected project files:

- `GraphIC.py` — IC graph representation, Monte Carlo reference implementation, and approximation methods.
- `utilities.py` — dataset loading and evaluation metrics.
- `ic_calibration.py` — edge-probability models and secondary-activation calibration.
- `synthetic_benchmark.py` — self-contained synthetic benchmark driver.
- `real_world_benchmark.py` — self-contained real-world benchmark driver.

The old exploratory modules (`synthetic_exploration.py`, `real_world_exploration.py`, and `spectral_ic_calibration.py`) are not required.


In [ ]:
from pathlib import Path
import runpy

import numpy as np

from GraphIC import (
    ICGraph,
    optimized_independent_cascade,
    swe_no,
    swe_no_cavity,
    swe,
    swe_cavity,
    swe_hib_cavity,
    additive_swe,
    additive_swe_cavity,
    Naive,
    dmp_est_r2,
    dmp_est,
    dmp_inf,
    modified_ALE,
    modified_ALE2,
    ALE_heuristic,
    ALE2,
    cavity_ALE,
    modified_ALE_cavity,
    SPM,
    SP1M,
    sss,
)

from utilities import (
    evaluate_predictions,
    load_dataset,
)

from ic_calibration import (
    make_base_probabilities,
    calibrate_secondary_activation_grid,
    edge_probability_summary,
)

import synthetic_benchmark as sb
import real_world_benchmark as rwb


## Fixed-parameter method adapter

The benchmark drivers handle parameter selection and runtime measurement. This adapter only maps each method name to the corresponding `GraphIC` implementation at a fixed parameter setting.


In [ ]:
def run_method_fixed(method, graph, prior_probs, params):
    T = params.get("T")
    layers = params.get("layers")

    if method == "dmp_est":
        return dmp_est(graph, prior_probs, T=T)

    if method == "dmp_est_r2":
        return dmp_est_r2(graph, prior_probs, T=T)

    if method == "swe":
        return swe(graph, prior_probs, T=T)

    if method == "swe_cavity":
        return swe_cavity(graph, prior_probs, T=T)

    if method == "additive_swe":
        return additive_swe(graph, prior_probs, num_steps=T)

    if method == "additive_swe_cavity":
        return additive_swe_cavity(graph, prior_probs, num_steps=T)

    if method == "swe_no":
        return swe_no(
            graph,
            prior_probs,
            T=T,
            a=1.0,
            layers=layers,
        )

    if method == "swe_no_cavity":
        return swe_no_cavity(
            graph,
            prior_probs,
            T=T,
            a=1.0,
            layers=layers,
        )

    if method == "swe_hib_cavity":
        return swe_hib_cavity(
            graph,
            prior_probs,
            T=T,
            eps=1e-20,
        )

    if method == "Naive":
        return Naive(
            graph,
            prior_probs,
            T=T,
            eps=1e-20,
        )

    if method == "ALE_heuristic":
        return ALE_heuristic(
            graph,
            prior_probs,
            num_steps=T,
        )

    if method == "ALE2":
        return ALE2(
            graph,
            prior_probs,
            num_steps=T,
        )

    if method == "modified_ALE":
        return modified_ALE(
            graph,
            prior_probs,
            num_steps=T,
        )

    # IPL2 in the paper; retained implementation name for compatibility.
    if method == "modified_ALE2":
        return modified_ALE2(
            graph,
            prior_probs,
            num_steps=T,
        )

    if method == "cavity_ALE":
        return cavity_ALE(
            graph,
            prior_probs,
            num_steps=T,
        )

    if method == "modified_ALE_cavity":
        return modified_ALE_cavity(
            graph,
            prior_probs,
            num_steps=T,
        )

    if method == "SPM":
        return SPM(graph, prior_probs)

    if method == "SP1M":
        return SP1M(graph, prior_probs)

    if method == "dmp_inf":
        return dmp_inf(
            graph,
            prior_probs,
            eps=1e-20,
            max_iter=100,
        )

    if method == "sss":
        return sss(
            graph,
            prior_probs,
            tol=1e-8,
            max_iter=100,
            eps=1e-12,
        )

    raise ValueError(f"Unknown benchmark method: {method}")


# Catch mismatches between the benchmark defaults and this adapter immediately.
_supported_methods = {
    "swe_no",
    "swe_no_cavity",
    "additive_swe",
    "additive_swe_cavity",
    "swe",
    "swe_cavity",
    "swe_hib_cavity",
    "Naive",
    "dmp_est_r2",
    "dmp_est",
    "dmp_inf",
    "modified_ALE",
    "modified_ALE2",
    "ALE_heuristic",
    "ALE2",
    "cavity_ALE",
    "modified_ALE_cavity",
    "SPM",
    "SP1M",
    "sss",
}

# Complete benchmark method list is defined below.


## Shared benchmark configuration

The values below reproduce the final experiment design: \(T\in\{1,\ldots,20\}\), continuation layers \(L\in\{0,\ldots,5\}\), three calibration instances, five held-out instances, three edge-probability models, four seed fractions, and three target secondary-activation levels.


In [ ]:
BASE_SEED = 42

T_VALUES = range(1, 21)
LAYER_VALUES = range(0, 6)

# Add the second-order ALE-family methods used in the paper.
# "modified_ALE2" is IPL2; the implementation keeps the historical function name.
BENCHMARK_METHODS = tuple(dict.fromkeys(
    tuple(sb.DEFAULT_METHODS)
    + ("ALE2", "modified_ALE2")
))

assert set(BENCHMARK_METHODS) <= _supported_methods


def build_benchmark_parameter_grids(module):
    """Build standard grids and add T-tuning for ALE2 and IPL2."""
    grids = module.build_default_parameter_grids(
        methods=BENCHMARK_METHODS,
        T_values=T_VALUES,
        layer_values=LAYER_VALUES,
    )
    grids["ALE2"] = [{"T": int(T)} for T in T_VALUES]
    grids["modified_ALE2"] = [{"T": int(T)} for T in T_VALUES]
    return grids


EDGE_MODELS = (
    "constant",
    "weighted_cascade",
    "heterogeneous_dirichlet",
)

SEED_FRACTIONS = (
    0.01,
    0.05,
    0.10,
    0.20,
)

TARGET_SECONDARY_ACTIVATIONS = (
    0.10,
    0.35,
    0.60,
)

DIRICHLET_CONCENTRATION = 0.5

NUM_CALIBRATION_INSTANCES = 3
NUM_TEST_INSTANCES = 5

PILOT_NUM_SIM = 250
BINARY_SEARCH_STEPS = 10
TARGET_TOL = 0.02

# These are intentionally explicit because running both benchmarks is expensive.
RUN_SYNTHETIC = True
RUN_REAL_WORLD = True

# Optional post-processing hook retained from the old notebook.
RUN_SYNTHETIC_ANALYSIS = False


## Synthetic benchmark

Calibration and held-out evaluation use independent synthetic graph realizations. Hyperparameters are selected using the calibration instances and then frozen for the held-out instances.


In [ ]:
synthetic_parameter_grids = build_benchmark_parameter_grids(sb)

synthetic_paths = None

if RUN_SYNTHETIC:
    synthetic_paths = sb.run_synthetic_benchmark(
        ICGraph=ICGraph,
        make_base_probabilities=make_base_probabilities,
        calibrate_secondary_activation_grid=calibrate_secondary_activation_grid,
        optimized_independent_cascade=optimized_independent_cascade,
        run_method_fixed=run_method_fixed,
        evaluate_predictions=evaluate_predictions,
        edge_probability_summary=edge_probability_summary,
        parameter_grids=synthetic_parameter_grids,
        output_dir="results/synthetic_benchmark",
        methods=BENCHMARK_METHODS,
        base_seed=BASE_SEED,
        n=5000,
        graph_families=(
            "ErdosRenyi",
            "BarabasiAlbert",
            "WattsStrogatz",
            "StochasticBlockModel",
            "HolmeKim",
        ),
        mean_degrees=(4, 10, 20),
        edge_models=EDGE_MODELS,
        dirichlet_concentration=DIRICHLET_CONCENTRATION,
        seed_fractions=SEED_FRACTIONS,
        target_secondary_activations=TARGET_SECONDARY_ACTIVATIONS,
        num_calibration_instances=NUM_CALIBRATION_INSTANCES,
        num_test_instances=NUM_TEST_INSTANCES,
        calibration_num_sim=10_000,
        test_num_sim=50_000,
        pilot_num_sim=PILOT_NUM_SIM,
        binary_search_steps=BINARY_SEARCH_STEPS,
        target_tol=TARGET_TOL,
        resume=True,
    )

synthetic_paths


### Optional synthetic-result analysis

This keeps the old notebook's analysis step without making `analyze_synthetic_benchmark.py` a required dependency of the benchmark itself.


In [ ]:
if RUN_SYNTHETIC_ANALYSIS:
    analysis_script = Path("analyze_synthetic_benchmark.py")
    if not analysis_script.exists():
        raise FileNotFoundError(
            "RUN_SYNTHETIC_ANALYSIS=True, but analyze_synthetic_benchmark.py "
            "was not found in the working directory."
        )
    runpy.run_path(str(analysis_script), run_name="__main__")


## Real-world benchmark

The graph topology is fixed within each dataset. Calibration and held-out instances use independent seed realizations and, for the heterogeneous condition, independent Dirichlet edge-weight realizations.


In [ ]:
real_parameter_grids = build_benchmark_parameter_grids(rwb)

real_paths = None

if RUN_REAL_WORLD:
    real_paths = rwb.run_real_world_benchmark(
        load_dataset=load_dataset,
        make_base_probabilities=make_base_probabilities,
        calibrate_secondary_activation_grid=calibrate_secondary_activation_grid,
        optimized_independent_cascade=optimized_independent_cascade,
        run_method_fixed=run_method_fixed,
        evaluate_predictions=evaluate_predictions,
        edge_probability_summary=edge_probability_summary,
        parameter_grids=real_parameter_grids,
        output_dir="results/real_world_benchmark",
        methods=BENCHMARK_METHODS,
        edge_models=EDGE_MODELS,
        dirichlet_concentration=DIRICHLET_CONCENTRATION,
        seed_fractions=SEED_FRACTIONS,
        target_secondary_activations=TARGET_SECONDARY_ACTIVATIONS,
        num_calibration_instances=NUM_CALIBRATION_INSTANCES,
        num_test_instances=NUM_TEST_INSTANCES,
        calibration_mc=rwb.DEFAULT_CALIBRATION_MC,
        test_mc=rwb.DEFAULT_TEST_MC,
        pilot_num_sim=PILOT_NUM_SIM,
        binary_search_steps=BINARY_SEARCH_STEPS,
        target_tol=TARGET_TOL,
        base_seed=BASE_SEED,
        resume=True,
    )

real_paths


## Outputs

Each benchmark writes:

- `graphs.csv`
- `instances.csv`
- `calibration_curves.csv`
- `selected_hyperparameters.csv`
- `benchmark_results.csv`

under its corresponding directory in `results/`.

Both benchmark drivers support `resume=True`, so an interrupted run can continue from already written results.
